# Chapter 3.1 Toy Flow Matching from Scratch

This split isolates the Chapter 3 toy experiment from the retired monolith. The tutorial question is: how does local conditional velocity regression become a learned marginal vector field that can move a source population into a target population?

The toy setting is deliberately two dimensional. Every object - endpoint pairs, conditional paths, velocity targets, the learned vector field, and generated samples - can be inspected before the EB notebooks reuse the same CFM hierarchy in a higher-dimensional biological state space.


## Tutorial setup

These cells set up imports, reproducible run modes, output directories, and generic save/display helpers. The CFM-specific data generation, training, sampling, and plotting logic remains visible in later tutorial cells.


In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

try:
    import torch
except ImportError as exc:
    raise ImportError("This notebook requires PyTorch for CFM training and sampling.") from exc

CWD = Path.cwd().resolve()
root_candidates = [
    CWD,
    CWD.parent,
    CWD.parent.parent,
    CWD / "flow_matching_for_dynamic_biology",
    CWD.parent / "flow_matching_for_dynamic_biology",
    CWD.parent.parent / "flow_matching_for_dynamic_biology",
]
for candidate in root_candidates:
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate.resolve()
        break
else:
    raise FileNotFoundError("Could not locate project root containing src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import ch03_tutorial as ch03
from src.utils import set_seed
from src.models import VelocityMLP, count_parameters
from src.train import train_cfm_steps
from src.sampling import euler_sample


In [ ]:
SEED = int(os.environ.get("CH03_SEED", "42"))
set_seed(SEED)
rng = np.random.default_rng(SEED)

QUICK_MODE = os.environ.get("CH03_QUICK", "1") == "1"
SMOKE_MODE = os.environ.get("CH03_SMOKE_MODE", "0") == "1"
PAPER_FIGURE_MODE = os.environ.get("CH03_PAPER_FIGURE_MODE", "1") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FULL_RUN_HINTS = {
    "eb_train_steps": 10000,
    "cnf_steps": 600,
    "time_ablation_steps": 1200,
    "capacity_steps": 1200,
    "notes": "Set CH03_QUICK=0 for full-run settings; default remains quick and reproducible.",
}

context = ch03.make_ch03_context(PROJECT_ROOT)
FIG_DIR = context.fig_dir
TABLE_DIR = context.table_dir
OUT_DIR = context.output_dir
PAPER_COLORS = ch03.PAPER_COLORS

figures_written = []
paper_ready_png_written = []
paper_ready_pdf_written = []
tables_written = []
paper_tables_written = []
skipped_items = []

ch03.set_paper_style()
run_mode = pd.DataFrame(
    [
        {"setting": "project_root", "value": str(PROJECT_ROOT)},
        {"setting": "device", "value": DEVICE},
        {"setting": "seed", "value": SEED},
        {"setting": "quick_mode", "value": QUICK_MODE},
        {"setting": "smoke_mode", "value": SMOKE_MODE},
        {"setting": "paper_figure_mode", "value": PAPER_FIGURE_MODE},
    ]
)
display(run_mode)


In [ ]:
add_panel_label = ch03.add_panel_label
short_strategy_label = ch03.short_strategy_label
clean_spines = ch03.clean_spines
format_metric_axis = ch03.format_metric_axis
add_note = ch03.add_note
as_np = ch03.as_np
subsample_idx = ch03.subsample_idx
robust_limits = ch03.robust_limits
format_axis = ch03.format_axis


from src.artifacts import safe_relpath


def _rel(path):
    return safe_relpath(path, root=PROJECT_ROOT)


def save_figure(fig, filename, dpi=300, write_pdf=None):
    if write_pdf is None:
        write_pdf = PAPER_FIGURE_MODE
    png_path = ch03.save_figure(fig, FIG_DIR, filename, dpi=dpi, write_pdf=write_pdf)
    rel_png = _rel(png_path)
    if rel_png not in figures_written:
        figures_written.append(rel_png)
    if write_pdf:
        rel_pdf = _rel(png_path.with_suffix(".pdf"))
        if rel_pdf not in paper_ready_pdf_written:
            paper_ready_pdf_written.append(rel_pdf)
    plt.close(fig)
    return png_path


def save_run_json(payload, filename):
    return ch03.save_json(OUT_DIR / filename, payload)


In [ ]:
def save_csv(frame, filename):
    path = ch03.save_csv(TABLE_DIR / filename, pd.DataFrame(frame))
    rel = _rel(path)
    if rel not in tables_written:
        tables_written.append(rel)
    return path


def display_saved_figure(path, width=None):
    return ch03.display_saved_figure(path, width=width)


def display_saved_figures(paths, width=None):
    return ch03.display_saved_figures(paths, width=width)


def display_table(frame, columns=None, n=10):
    return ch03.display_table(pd.DataFrame(frame), columns=columns, n=n)


## Artifact provenance from the retired monolith

The retired `03_flow_matching_from_scratch.ipynb` bundled toy, EB, baseline, ablation, and audit sections together. This split owns only the old toy section and its four key figure files. Later Chapter 3 notebooks own EB training, solver diagnostics, baselines, ablations, and claim audits.


In [ ]:
toy_artifact_contract = pd.DataFrame(
    [
        {
            "artifact": "fig_toy_loss.png",
            "retired_source": "Section 1 / 2D Toy Sanity Check",
            "purpose": "local CFM regression loss",
        },
        {
            "artifact": "fig_toy_evolution.png",
            "retired_source": "Section 1 / 2D Toy Sanity Check",
            "purpose": "Euler rollout of learned toy field",
        },
        {
            "artifact": "fig03_02_conditional_vs_marginal_toy.png",
            "retired_source": "Fig 3.2 / conditional versus marginal",
            "purpose": "conditional path velocities versus learned marginal velocity",
        },
        {
            "artifact": "fig03_03_cfm_object_hierarchy_toy.png",
            "retired_source": "Fig 3.3 / object hierarchy",
            "purpose": "endpoint path, path population, learned marginal field",
        },
    ]
)
display_table(toy_artifact_contract, n=len(toy_artifact_contract))


## 1. 2D Toy Sanity Check

The source distribution is eight small Gaussian components on a ring. The target distribution is one rotated Gaussian near the center. CFM never observes a true trajectory here; it samples random endpoint pairs and learns from straight conditional paths between them.


In [ ]:
def make_eight_gaussians(n=1500, radius=2.0, noise=0.08, seed=42):
    local_rng = np.random.default_rng(seed)
    angles = np.linspace(0.0, 2.0 * np.pi, 8, endpoint=False)
    centers = np.column_stack([radius * np.cos(angles), radius * np.sin(angles)])
    component = local_rng.integers(0, 8, size=n)
    points = centers[component] + local_rng.normal(scale=noise, size=(n, 2))
    return points.astype(np.float32), component


def make_single_gaussian(n=1500, loc=(0.0, 0.0), scale=(0.42, 0.28), angle=0.35, seed=43):
    local_rng = np.random.default_rng(seed)
    raw = local_rng.normal(size=(n, 2)).astype(np.float32)
    c, s = np.cos(angle), np.sin(angle)
    R = np.array([[c, -s], [s, c]], dtype=np.float32)
    S = np.diag(np.asarray(scale, dtype=np.float32))
    points = raw @ S @ R.T + np.asarray(loc, dtype=np.float32)
    return points.astype(np.float32)


def make_random_pair_batch_fn(X0, X1, seed=42):
    local_rng = np.random.default_rng(seed)

    def pair_batch_fn(batch_size):
        idx0 = local_rng.integers(0, len(X0), size=int(batch_size))
        idx1 = local_rng.integers(0, len(X1), size=int(batch_size))
        return {"x0": X0[idx0].astype(np.float32), "x1": X1[idx1].astype(np.float32)}

    return pair_batch_fn


In [ ]:
toy_n = 1500 if QUICK_MODE else 5000
if SMOKE_MODE:
    toy_n = 350

toy_steps = 550 if QUICK_MODE else 2200
if SMOKE_MODE:
    toy_steps = 35
toy_log_every = 25 if not SMOKE_MODE else 5

toy_design = pd.DataFrame(
    [
        {"quantity": "toy samples per endpoint distribution", "value": toy_n},
        {"quantity": "training steps", "value": toy_steps},
        {"quantity": "training log interval", "value": toy_log_every},
        {"quantity": "training batch size", "value": 256 if not SMOKE_MODE else 96},
    ]
)
display_table(toy_design, n=len(toy_design))


In [ ]:
toy_X0, toy_components = make_eight_gaussians(n=toy_n, seed=SEED)
toy_X1 = make_single_gaussian(n=toy_n, seed=SEED + 1)
toy_pair_batch_fn = make_random_pair_batch_fn(toy_X0, toy_X1, seed=SEED + 2)

distribution_summary = pd.DataFrame(
    [
        {
            "distribution": "source eight-gaussian mixture",
            "n": len(toy_X0),
            "mean_x": toy_X0[:, 0].mean(),
            "mean_y": toy_X0[:, 1].mean(),
            "std_x": toy_X0[:, 0].std(),
            "std_y": toy_X0[:, 1].std(),
            "components": int(np.unique(toy_components).size),
        },
        {
            "distribution": "target rotated Gaussian",
            "n": len(toy_X1),
            "mean_x": toy_X1[:, 0].mean(),
            "mean_y": toy_X1[:, 1].mean(),
            "std_x": toy_X1[:, 0].std(),
            "std_y": toy_X1[:, 1].std(),
            "components": 1,
        },
    ]
)
display_table(distribution_summary, n=len(distribution_summary))


## 2. Conditional path and local velocity target

For a sampled endpoint pair `(x0, x1)` and time `t`, the conditional path is `x_t = (1 - t) x0 + t x1`. The local regression target is the straight-path velocity `u_t = x1 - x0`. This is conditional because it belongs to one sampled pair; the marginal velocity is what remains after averaging over many possible endpoint pairs that pass near the same state and time.


In [ ]:
preview_rng = np.random.default_rng(SEED + 101)
preview_n = 6
preview_i0 = preview_rng.integers(0, len(toy_X0), size=preview_n)
preview_i1 = preview_rng.integers(0, len(toy_X1), size=preview_n)
preview_t = np.linspace(0.1, 0.9, preview_n, dtype=np.float32)
preview_x0 = toy_X0[preview_i0]
preview_x1 = toy_X1[preview_i1]
preview_xt = (1.0 - preview_t[:, None]) * preview_x0 + preview_t[:, None] * preview_x1
preview_ut = preview_x1 - preview_x0

conditional_path_preview = pd.DataFrame(
    {
        "t": preview_t,
        "x0_1": preview_x0[:, 0],
        "x0_2": preview_x0[:, 1],
        "x1_1": preview_x1[:, 0],
        "x1_2": preview_x1[:, 1],
        "x_t_1": preview_xt[:, 0],
        "x_t_2": preview_xt[:, 1],
        "u_t_1": preview_ut[:, 0],
        "u_t_2": preview_ut[:, 1],
    }
)
display_table(conditional_path_preview.round(3), n=preview_n)


## 3. Learn the marginal velocity field

Training repeatedly samples endpoint pairs and times, computes the conditional target `x1 - x0`, and updates `VelocityMLP(x_t, t)` by local mean-squared error. The network is the marginal field estimator: it cannot memorize one endpoint path, so it must represent the local average behavior induced by many paths.


In [ ]:
toy_model = VelocityMLP(x_dim=2, hidden_dim=128, hidden_layers=4).to(DEVICE)
toy_optimizer = torch.optim.Adam(toy_model.parameters(), lr=1e-3)

model_summary = pd.DataFrame(
    [
        {"object": "VelocityMLP", "role": "marginal vector-field estimator"},
        {"object": "input x_t", "role": "current 2D state on a conditional path"},
        {"object": "input t", "role": "continuous interpolation time"},
        {"object": "output v_theta(x_t,t)", "role": "predicted local velocity"},
        {"object": "parameters", "role": int(count_parameters(toy_model))},
    ]
)
display_table(model_summary, n=len(model_summary))


In [ ]:
toy_history = train_cfm_steps(
    toy_model,
    toy_pair_batch_fn,
    toy_optimizer,
    n_steps=toy_steps,
    batch_size=256 if not SMOKE_MODE else 96,
    device=DEVICE,
    log_every=toy_log_every,
)


In [ ]:
loss_preview = toy_history.tail(8).copy()
display_table(loss_preview, n=len(loss_preview))


In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.plot(toy_history["step"], toy_history["loss"], color="#2F6B5E", linewidth=1.6)
ax.set_xlabel("optimization step")
ax.set_ylabel("CFM local MSE")
ax.set_title("Toy CFM training loss")
ax.grid(axis="y", color="0.92", linewidth=0.7)
toy_loss_path = save_figure(fig, "fig_toy_loss.png")


In [ ]:
display_saved_figure(toy_loss_path, width=620)


## 4. Roll out the learned field

Sampling now uses the learned marginal vector field as an ODE velocity. Euler integration starts from source particles and repeatedly applies `dx/dt = v_theta(x,t)`. This checks whether local CFM regression learned a population-level transport map in the toy space.


In [ ]:
toy_model.eval()
toy_sample_n = min(900 if QUICK_MODE else 1800, len(toy_X0))
if SMOKE_MODE:
    toy_sample_n = min(220, len(toy_X0))

toy_sample_idx = subsample_idx(len(toy_X0), toy_sample_n, seed=SEED + 3)
toy_x0_t = torch.as_tensor(toy_X0[toy_sample_idx], dtype=torch.float32, device=DEVICE)
toy_euler_steps = 100 if not SMOKE_MODE else 25

sampling_design = pd.DataFrame(
    [
        {"quantity": "sampled source particles", "value": toy_sample_n},
        {"quantity": "Euler steps", "value": toy_euler_steps},
        {"quantity": "return full trajectory", "value": True},
    ]
)
display_table(sampling_design, n=len(sampling_design))


In [ ]:
with torch.no_grad():
    toy_final_t, toy_traj_t, toy_nfe = euler_sample(
        toy_model,
        toy_x0_t,
        n_steps=toy_euler_steps,
        return_traj=True,
    )
toy_traj = as_np(toy_traj_t)


In [ ]:
trajectory_summary = pd.DataFrame(
    [
        {"quantity": "trajectory tensor shape", "value": str(tuple(toy_traj.shape))},
        {"quantity": "Euler NFE", "value": int(toy_nfe)},
        {"quantity": "initial mean x", "value": float(toy_traj[0, :, 0].mean())},
        {"quantity": "final mean x", "value": float(toy_traj[-1, :, 0].mean())},
        {"quantity": "initial mean y", "value": float(toy_traj[0, :, 1].mean())},
        {"quantity": "final mean y", "value": float(toy_traj[-1, :, 1].mean())},
    ]
)
display_table(trajectory_summary, n=len(trajectory_summary))


In [ ]:
toy_times = [0.0, 0.25, 0.5, 0.75, 1.0]
toy_step_idx = np.round(np.asarray(toy_times) * (toy_traj.shape[0] - 1)).astype(int)
xlim, ylim = robust_limits(toy_X0, toy_X1, toy_traj.reshape(-1, 2), margin=0.10)
fig, axes = plt.subplots(1, 5, figsize=(12.3, 2.65), sharex=True, sharey=True)
target_idx = subsample_idx(len(toy_X1), 450, seed=61)
for ax, tau, step in zip(axes, toy_times, toy_step_idx):
    pts = toy_traj[step]
    ax.scatter(toy_X1[target_idx, 0], toy_X1[target_idx, 1], s=5, color="0.72", alpha=0.20, linewidths=0)
    ax.scatter(pts[:, 0], pts[:, 1], s=6, color="#2F6B5E", alpha=0.52, linewidths=0)
    format_axis(ax, xlim, ylim, xlabel="toy x1", ylabel="toy x2", title=f"t={tau:.2f}")
    if ax is not axes[0]:
        ax.set_ylabel("")
fig.suptitle("Toy population evolution under Euler sampling")
toy_evolution_path = save_figure(fig, "fig_toy_evolution.png")


In [ ]:
display_saved_figure(toy_evolution_path, width=900)


### Figure 3.2: Conditional Velocity Versus Marginal Velocity

This panel inspects the distinction that CFM relies on. Around one local state at `t=0.5`, many conditional endpoint pairs imply different straight-line velocities. Their local average is the Monte Carlo target for the marginal velocity field. The trained network predicts one vector at that state and time.


In [ ]:
def build_toy_velocity_probe(model, X0, X1, device="cpu", seed=42):
    local_rng = np.random.default_rng(seed)
    n_pairs = min(1800, len(X0), len(X1))
    i0 = local_rng.integers(0, len(X0), size=n_pairs)
    i1 = local_rng.integers(0, len(X1), size=n_pairs)
    x0 = X0[i0]
    x1 = X1[i1]
    t_value = 0.5
    x_t = (1.0 - t_value) * x0 + t_value * x1
    u_t = x1 - x0

    center = np.median(x_t, axis=0)
    dist = np.linalg.norm(x_t - center[None, :], axis=1)
    radius = np.quantile(dist, 0.18)
    local = np.flatnonzero(dist <= max(radius, 1e-6))
    if len(local) > 120:
        local = local[subsample_idx(len(local), 120, seed=seed + 1)]

    model.eval()
    with torch.no_grad():
        center_t = torch.as_tensor(center[None, :], dtype=torch.float32, device=device)
        time_t = torch.full((1, 1), t_value, dtype=torch.float32, device=device)
        pred = model(center_t, time_t).detach().cpu().numpy()[0]
    mean_conditional = u_t[local].mean(axis=0)
    return {
        "x_t": x_t,
        "u_t": u_t,
        "local": local,
        "center": center,
        "t_value": t_value,
        "network_velocity": pred,
        "mean_conditional_velocity": mean_conditional,
    }


toy_velocity_probe = build_toy_velocity_probe(toy_model, toy_X0, toy_X1, device=DEVICE, seed=SEED + 4)


In [ ]:
velocity_probe_summary = pd.DataFrame(
    [
        {"object": "local conditional velocities", "value": int(len(toy_velocity_probe["local"]))},
        {"object": "probe time", "value": toy_velocity_probe["t_value"]},
        {"object": "probe center x", "value": toy_velocity_probe["center"][0]},
        {"object": "probe center y", "value": toy_velocity_probe["center"][1]},
        {"object": "mean conditional velocity x", "value": toy_velocity_probe["mean_conditional_velocity"][0]},
        {"object": "network velocity x", "value": toy_velocity_probe["network_velocity"][0]},
        {"object": "mean conditional velocity y", "value": toy_velocity_probe["mean_conditional_velocity"][1]},
        {"object": "network velocity y", "value": toy_velocity_probe["network_velocity"][1]},
    ]
)
display_table(velocity_probe_summary.round(4), n=len(velocity_probe_summary))


In [ ]:
def plot_toy_conditional_vs_marginal(model, X0, X1, probe):
    x_t = probe["x_t"]
    u_t = probe["u_t"]
    local = probe["local"]
    center = probe["center"]
    pred = probe["network_velocity"]
    mean_conditional = probe["mean_conditional_velocity"]

    xlim, ylim = robust_limits(X0, X1, x_t[local], margin=0.12)
    fig, ax = plt.subplots(figsize=(5.2, 4.5))
    source_idx = subsample_idx(len(X0), 550, seed=71)
    target_idx = subsample_idx(len(X1), 550, seed=72)
    ax.scatter(X0[source_idx, 0], X0[source_idx, 1], s=5, color="#4C78A8", alpha=0.18, linewidths=0, label="source")
    ax.scatter(X1[target_idx, 0], X1[target_idx, 1], s=5, color="#D1495B", alpha=0.18, linewidths=0, label="target")
    ax.scatter(x_t[local, 0], x_t[local, 1], s=11, color="0.55", alpha=0.35, linewidths=0)
    ax.quiver(x_t[local, 0], x_t[local, 1], u_t[local, 0], u_t[local, 1], angles="xy", scale_units="xy", scale=12, width=0.003, color="#9CC9C2", alpha=0.52, label="conditional velocities")
    ax.quiver([center[0]], [center[1]], [mean_conditional[0]], [mean_conditional[1]], angles="xy", scale_units="xy", scale=5, width=0.010, color="0.18", alpha=0.85, label="local conditional average")
    ax.quiver([center[0]], [center[1]], [pred[0]], [pred[1]], angles="xy", scale_units="xy", scale=5, width=0.013, color="#B9352B", alpha=0.95, label="network prediction")
    ax.scatter([center[0]], [center[1]], s=42, color="#B9352B", edgecolor="white", linewidth=0.7, zorder=5)
    format_axis(ax, xlim, ylim, xlabel="toy state 1", ylabel="toy state 2", title="Conditional velocities collapse to a marginal vector")
    ax.legend(frameon=False, loc="best")
    return fig


In [ ]:
fig = plot_toy_conditional_vs_marginal(toy_model, toy_X0, toy_X1, toy_velocity_probe)
toy_fig32_path = save_figure(fig, "fig03_02_conditional_vs_marginal_toy.png")


In [ ]:
display_saved_figure(toy_fig32_path, width=620)


### Figure 3.3: CFM Object Hierarchy

CFM has a hierarchy of objects. At the bottom is one endpoint-conditioned straight path. A batch gives many such paths. The model is not one of those paths; it is an Eulerian vector field that summarizes the marginal velocity at each state and time.


In [ ]:
def prepare_toy_hierarchy_objects(model, X0, X1, device="cpu", seed=42):
    local_rng = np.random.default_rng(seed)
    n_paths = min(90, len(X0), len(X1))
    i0 = local_rng.integers(0, len(X0), size=n_paths)
    i1 = local_rng.integers(0, len(X1), size=n_paths)
    x0 = X0[i0]
    x1 = X1[i1]
    xlim, ylim = robust_limits(X0, X1, x0, x1, margin=0.12)

    grid_n = 20
    xs = np.linspace(xlim[0], xlim[1], grid_n)
    ys = np.linspace(ylim[0], ylim[1], grid_n)
    gx, gy = np.meshgrid(xs, ys)
    pts = np.column_stack([gx.ravel(), gy.ravel()]).astype(np.float32)

    model.eval()
    with torch.no_grad():
        x_t = torch.as_tensor(pts, dtype=torch.float32, device=device)
        t_t = torch.full((pts.shape[0], 1), 0.5, dtype=torch.float32, device=device)
        v = model(x_t, t_t).detach().cpu().numpy()
    norm = np.linalg.norm(v, axis=1)
    cap = np.percentile(norm[norm > 0], 88) if np.any(norm > 0) else 1.0
    v = v * np.minimum(1.0, cap / np.clip(norm[:, None], 1e-12, None))
    return {"x0": x0, "x1": x1, "xlim": xlim, "ylim": ylim, "grid_points": pts, "grid_velocity": v}


toy_hierarchy = prepare_toy_hierarchy_objects(toy_model, toy_X0, toy_X1, device=DEVICE, seed=SEED + 5)


In [ ]:
hierarchy_preview = pd.DataFrame(
    [
        {"level": "one endpoint pair", "object": "x0, x1", "visible_in_panel": "A"},
        {"level": "one conditional path", "object": "x_t = (1 - t)x0 + t x1", "visible_in_panel": "A"},
        {"level": "path population", "object": "many independently sampled endpoint pairs", "visible_in_panel": "B"},
        {"level": "marginal velocity field", "object": "v_theta(x,t) over a grid at t=0.5", "visible_in_panel": "C"},
    ]
)
display_table(hierarchy_preview, n=len(hierarchy_preview))


In [ ]:
def plot_toy_cfm_object_hierarchy(X0, X1, hierarchy):
    x0 = hierarchy["x0"]
    x1 = hierarchy["x1"]
    xlim = hierarchy["xlim"]
    ylim = hierarchy["ylim"]
    pts = hierarchy["grid_points"]
    v = hierarchy["grid_velocity"]

    fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.8), sharex=True, sharey=True)
    source_idx = subsample_idx(len(X0), 500, seed=81)
    target_idx = subsample_idx(len(X1), 500, seed=82)
    for ax in axes:
        ax.scatter(X0[source_idx, 0], X0[source_idx, 1], s=5, color="#4C78A8", alpha=0.14, linewidths=0)
        ax.scatter(X1[target_idx, 0], X1[target_idx, 1], s=5, color="#D1495B", alpha=0.14, linewidths=0)
        format_axis(ax, xlim, ylim, xlabel="toy state 1", ylabel="toy state 2")

    a, b = x0[0], x1[0]
    path = np.stack([(1.0 - t) * a + t * b for t in np.linspace(0, 1, 50)], axis=0)
    axes[0].plot(path[:, 0], path[:, 1], color="0.18", linewidth=1.6)
    axes[0].scatter([a[0], b[0]], [a[1], b[1]], s=40, color=["#4C78A8", "#D1495B"], edgecolor="white", linewidth=0.7, zorder=4)
    mid = 0.5 * (a + b)
    vel = b - a
    axes[0].quiver([mid[0]], [mid[1]], [vel[0]], [vel[1]], angles="xy", scale_units="xy", scale=4, width=0.012, color="#2F6B5E")
    axes[0].set_title("A. one endpoint path")

    for j in range(len(x0)):
        axes[1].plot([x0[j, 0], x1[j, 0]], [x0[j, 1], x1[j, 1]], color="0.25", alpha=0.18, linewidth=0.65)
    axes[1].set_title("B. many endpoint paths")
    axes[1].set_ylabel("")

    axes[2].quiver(pts[:, 0], pts[:, 1], v[:, 0], v[:, 1], angles="xy", scale_units="xy", scale=16, width=0.004, color="#B9352B", alpha=0.72)
    axes[2].set_title("C. learned marginal field at t=0.5")
    axes[2].set_ylabel("")
    fig.suptitle("CFM object hierarchy on a 2D toy problem")
    return fig


In [ ]:
fig = plot_toy_cfm_object_hierarchy(toy_X0, toy_X1, toy_hierarchy)
toy_hierarchy_path = save_figure(fig, "fig03_03_cfm_object_hierarchy_toy.png")


In [ ]:
display_saved_figure(toy_hierarchy_path, width=900)


## Artifact audit

The final cell defines this notebook's output contract, writes the run configuration, records the artifact manifest, and fails loudly if any expected figure or output is missing or empty. There are no persisted toy tables in this split; the preview tables above are tutorial displays only.


In [ ]:
expected_figures = [
    FIG_DIR / "fig_toy_loss.png",
    FIG_DIR / "fig_toy_evolution.png",
    FIG_DIR / "fig03_02_conditional_vs_marginal_toy.png",
    FIG_DIR / "fig03_03_cfm_object_hierarchy_toy.png",
]
expected_tables = []
run_config_path = OUT_DIR / "run_config_03_1_toy_flow_matching_from_scratch.json"
manifest_path = OUT_DIR / "artifact_manifest_03_1_toy_flow_matching_from_scratch.csv"
expected_outputs = [run_config_path, manifest_path]

save_run_json(
    {
        "seed": SEED,
        "quick_mode": QUICK_MODE,
        "smoke_mode": SMOKE_MODE,
        "paper_figure_mode": PAPER_FIGURE_MODE,
        "owned_figures": [path.name for path in expected_figures],
        "full_run_hints": FULL_RUN_HINTS,
    },
    run_config_path.name,
)

artifact_manifest = ch03.check_required_artifacts(
    expected_figures=expected_figures,
    expected_tables=expected_tables,
    expected_outputs=[run_config_path],
)
ch03.save_csv(manifest_path, artifact_manifest)
artifact_manifest = ch03.check_required_artifacts(
    expected_figures=expected_figures,
    expected_tables=expected_tables,
    expected_outputs=expected_outputs,
)
ch03.save_csv(manifest_path, artifact_manifest)
artifact_manifest = ch03.check_required_artifacts(
    expected_figures=expected_figures,
    expected_tables=expected_tables,
    expected_outputs=expected_outputs,
)
if not artifact_manifest["exists"].all():
    raise FileNotFoundError("Missing toy Flow Matching artifacts")

display_table(artifact_manifest, n=len(artifact_manifest))
